In [1]:
# =============================================================================
# Antibiotic Resistance Prediction using Proximity, Frequency, and G-score
# =============================================================================
# 
# This notebook trains and evaluates Random Forest classifiers on mutation-level
# features (mutation frequency, 3D structural proximity to known resistance-conferring
# mutations, and g-scores) to predict antibiotic resistance in Mycobacterium tuberculosis.
# It includes baseline modeling,combined modeling and predictions for uncertain mutations (Category 3).

In [1]:
## load dependencies
import pandas as pd
import numpy as np
import re
from evcouplings.compare import DistanceMap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score,roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay)
from scipy import stats
import os,json

In [2]:
from utils import *

## data preprocessing

In [3]:
# ----------------------------
# STEP 1: Load and Inspect Dataset
# ----------------------------
final_df = pd.read_csv("../data/derived_features/all_proteins_freq_proximity_gscore.csv")
print("Unique genes:", np.unique(final_df['gene']))
print("Data shape:", final_df.shape)

Unique genes: ['Rv0678' 'atpE' 'ddn' 'embB' 'ethA' 'fgd1' 'gid' 'gyrA' 'gyrB' 'inhA'
 'katG' 'pepQ' 'pncA' 'rplC' 'rpoB' 'rpsL' 'tlyA']
Data shape: (4690, 14)


In [4]:
# STEP 2: Basic Preprocessing
# ----------------------------
# Separate Category 3 entries (uncertain mutations)
category_3_data = final_df[final_df['confidence'] == "3) Uncertain significance"]

# Retain only labeled data (non-Category 3) for supervised learning
label_data = final_df[final_df['confidence'] != "3) Uncertain significance"]

# Target conversion to binary
label_data['binary_confidence'] = label_data['confidence'].map({
    "1) Assoc w R": 0,
    "2) Assoc w R - Interim": 0,
    "4) Not assoc w R - Interim": 1,
    "5) Not assoc w R": 1
})

# Drop missing targets
label_data = label_data[label_data['binary_confidence'].notna()]

#some debugging steps
print(label_data['confidence'].value_counts())
print(np.unique(label_data['confidence']))
# Identify rows where the 'frequency' column is NaN
nan_frequency_rows = label_data[label_data['frequency'].isna()]

diverse_sample_mutation_table = label_data.groupby('frequency').first().reset_index()
diverse_sample_mutation_table.head()

confidence
2) Assoc w R - Interim        160
1) Assoc w R                  155
5) Not assoc w R               38
4) Not assoc w R - Interim     22
Name: count, dtype: int64
['1) Assoc w R' '2) Assoc w R - Interim' '4) Not assoc w R - Interim'
 '5) Not assoc w R']


/tmp/ipykernel_191385/3132060444.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label_data['binary_confidence'] = label_data['confidence'].map({


,frequency,drug,gene,variant,confidence,one_letter_mutation,phenotype,position,WHO_Adjusted_Position,Proximity_1D,Nearest_1D_Index,Proximity_to_R_Conferring,Nearest_Mutation_Index,G_score,binary_confidence
0,0.000000,RIF,rpoB,rpoB_G426S,2) Assoc w R - Interim,G426S,Resistant,426,432.0,1.0,433.0,1.335177,433.0,2.296259,0
1,0.010204,EMB,embB,embB_Q139H,5) Not assoc w R,Q139H,Susceptible,139,NaN,65.0,74.0,32.089950,74.0,-0.302846,1
2,0.013274,LEV,gyrB,gyrB_P94L,5) Not assoc w R,P94L,Susceptible,94,NaN,367.0,461.0,57.169795,501.0,-1.423563,1
3,0.018182,RIF,rpoB,rpoB_K944E,5) Not assoc w R,K944E,Susceptible,944,950.0,453.0,497.0,44.237799,449.0,-0.120555,1
4,0.020609,MXF,gyrA,gyrA_A384V,5) Not assoc w R,A384V,Susceptible,384,NaN,290.0,94.0,39.790840,91.0,-0.177343,1


## run the baseline and combined models

In [5]:
feature_sets = {
    "Frequency": ['frequency'],
    "Proximity": ['Proximity_to_R_Conferring'],
    "Proximity_1d":["Proximity_1D"],
    "G-score": ['G_score'],
    "Frequency + Proximity": ['frequency', 'Proximity_to_R_Conferring'],
    "Frequency + G-score": ['frequency', 'G_score'],
    "Combined": ['frequency', 'Proximity_to_R_Conferring', 'G_score']
}


In [6]:
results = {}

for label, numeric_columns in feature_sets.items():
    print(f"\n=== Running model for: {label} ===")

    # Step 3: Feature selection
    label_data[numeric_columns] = label_data[numeric_columns].fillna(0)
    
    # Step 4: Split
    X_train, X_test, y_train, y_test, train_indices, test_indices = data_split(label_data, numeric_columns)
    
    # Step 5: Train and evaluate
    trained_rf, y_pred_test, y_prob_test = model_training(X_train, y_train, X_test)
    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_test, labels=[0,1]) #0-resistant
    print("Confusion matrix", cm)
    tn,fp,fn, tp = cm.ravel()
    sensitivity = tp / (tp+fn) #sensitivity resistant
    specificity = tn / (tn+fp) # specificity resistant
    print(f" Sensitivity: {sensitivity}, specificity: {specificity}")
    accuracy, roc_auc, precision = performance_evaluation(y_test, y_pred_test, y_prob_test)
    print("Accuracy:", accuracy)
    print("AUC:", roc_auc)
    print("Precision:", precision)

    # Step 6: Feature importance
    # feature_importances = plot_feature_importance(trained_rf, numeric_columns)
    feature_importances = trained_rf.feature_importances_

    # Step 7: Category 3 prediction
    (
        misclassified_original_indices,
        misclassified_samples,
        confidence_0_count,
        confidence_0_percentage,
        confidence_1_count,
        confidence_1_percentage
    ) = predict_category_3(
        category_3_data,
        trained_rf,
        final_df,
        numeric_columns,
        y_test,
        y_pred_test,
        test_indices
    )

    # Store all results
    results[label] = {
        "features": numeric_columns,
        "accuracy": accuracy,
        "precision": precision,
        "roc_auc": roc_auc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "feature_importance": feature_importances,
        "misclassified_indices": misclassified_original_indices,
        "misclassified_samples": misclassified_samples,
        "category_3_predictions": {
            "resistant_count": confidence_0_count,
            "susceptible_count": confidence_1_count,
            "resistant_percentage": confidence_0_percentage,
            "susceptible_percentage": confidence_1_percentage
        }
    }



=== Running model for: Frequency ===
Confusion Matrix:
Confusion matrix [[88  7]
 [ 0 18]]
 Sensitivity: 1.0, specificity: 0.9263157894736842
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.93      0.96        95
           1       0.72      1.00      0.84        18

    accuracy                           0.94       113
   macro avg       0.86      0.96      0.90       113
weighted avg       0.96      0.94      0.94       113

Accuracy: 0.9380530973451328
AUC: {0: 0.9894736842105264, 1: 0.9894736842105263}
Precision: 0.9553982300884956


/work/pi_annagreen_umass_edu/mahbuba/MTB_Mut_Clust/scripts/utils.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  category_3_data[numeric_columns] = category_3_data[numeric_columns].fillna(0)
/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/work/pi_annagreen_umass_edu/mahbuba/MTB_Mut_Clust/scripts/utils.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a


=== Running model for: Proximity ===
Confusion Matrix:
Confusion matrix [[95  0]
 [ 6 12]]
 Sensitivity: 0.6666666666666666, specificity: 1.0
Classification Report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        95
           1       1.00      0.67      0.80        18

    accuracy                           0.95       113
   macro avg       0.97      0.83      0.88       113
weighted avg       0.95      0.95      0.94       113

Accuracy: 0.9469026548672567
AUC: {0: 0.860233918128655, 1: 0.8602339181286549}
Precision: 0.9500569525979147


/work/pi_annagreen_umass_edu/mahbuba/MTB_Mut_Clust/scripts/utils.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  category_3_data[numeric_columns] = category_3_data[numeric_columns].fillna(0)
/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/work/pi_annagreen_umass_edu/mahbuba/MTB_Mut_Clust/scripts/utils.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a


=== Running model for: Proximity_1d ===
Confusion Matrix:
Confusion matrix [[95  0]
 [ 9  9]]
 Sensitivity: 0.5, specificity: 1.0
Classification Report:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        95
           1       1.00      0.50      0.67        18

    accuracy                           0.92       113
   macro avg       0.96      0.75      0.81       113
weighted avg       0.93      0.92      0.91       113

Accuracy: 0.9203539823008849
AUC: {0: 0.8833333333333333, 1: 0.8833333333333335}
Precision: 0.9272464261402313


...

/work/pi_annagreen_umass_edu/mahbuba/MTB_Mut_Clust/scripts/utils.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  category_3_data[numeric_columns] = category_3_data[numeric_columns].fillna(0)
/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/work/pi_annagreen_umass_edu/mahbuba/MTB_Mut_Clust/scripts/utils.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a


=== Running model for: Frequency + G-score ===
Confusion Matrix:
Confusion matrix [[93  2]
 [ 0 18]]
 Sensitivity: 1.0, specificity: 0.9789473684210527
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99        95
           1       0.90      1.00      0.95        18

    accuracy                           0.98       113
   macro avg       0.95      0.99      0.97       113
weighted avg       0.98      0.98      0.98       113

Accuracy: 0.9823008849557522
AUC: {0: 0.9976608187134504, 1: 0.9976608187134502}
Precision: 0.984070796460177


/work/pi_annagreen_umass_edu/mahbuba/MTB_Mut_Clust/scripts/utils.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  category_3_data[numeric_columns] = category_3_data[numeric_columns].fillna(0)
/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/work/pi_annagreen_umass_edu/mahbuba/MTB_Mut_Clust/scripts/utils.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a


=== Running model for: Combined ===
Confusion Matrix:
Confusion matrix [[94  1]
 [ 2 16]]
 Sensitivity: 0.8888888888888888, specificity: 0.9894736842105263
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98        95
           1       0.94      0.89      0.91        18

    accuracy                           0.97       113
   macro avg       0.96      0.94      0.95       113
weighted avg       0.97      0.97      0.97       113

Accuracy: 0.9734513274336283
AUC: {0: 0.9964912280701754, 1: 0.9964912280701754}
Precision: 0.9731151310081554


/work/pi_annagreen_umass_edu/mahbuba/MTB_Mut_Clust/scripts/utils.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  category_3_data[numeric_columns] = category_3_data[numeric_columns].fillna(0)
/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/work/pi_annagreen_umass_edu/mahbuba/MTB_Mut_Clust/scripts/utils.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a

In [7]:
def find_invalid_keys(obj, path="root"):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if not isinstance(k, (str, int, float, bool, type(None))):
                print(f"[Invalid Key] Path: {path} → Key: {k} (type: {type(k).__name__})")
            find_invalid_keys(v, path=f"{path}['{k}']")
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            find_invalid_keys(item, path=f"{path}[{i}]")


# Check before saving
find_invalid_keys(results)

def convert_keys_to_str(obj):
    if isinstance(obj, dict):
        return {str(k): convert_keys_to_str(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_keys_to_str(i) for i in obj]
    else:
        return obj
cleaned_results = convert_keys_to_str(results)

[Invalid Key] Path: root['Frequency']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path: root['Frequency']['roc_auc'] → Key: 1 (type: int64)
[Invalid Key] Path: root['Proximity']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path: root['Proximity']['roc_auc'] → Key: 1 (type: int64)
[Invalid Key] Path: root['Proximity_1d']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path: root['Proximity_1d']['roc_auc'] → Key: 1 (type: int64)
[Invalid Key] Path: root['G-score']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path: root['G-score']['roc_auc'] → Key: 1 (type: int64)
[Invalid Key] Path: root['Frequency + Proximity']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path: root['Frequency + Proximity']['roc_auc'] → Key: 1 (type: int64)
[Invalid Key] Path: root['Frequency + G-score']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path: root['Frequency + G-score']['roc_auc'] → Key: 1 (type: int64)
[Invalid Key] Path: root['Combined']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path:

In [8]:
with open("../results/baseline_combined_results.json", "w") as f:
    for label in cleaned_results:
        for key in cleaned_results[label]:
            if not isinstance(key, (str, int, float, bool, type(None))):
                print("Invalid key type:", key, type(key))
    json.dump(cleaned_results, f, indent=4, default=str)

print("\nAll model combinations evaluated and saved.")


All model combinations evaluated and saved.


In [9]:
def generate_markdown_table(results_dict):
    table_md = "| Model | Accuracy | AUC (avg) | Precision | Sensitivity | Specificity\n"
    table_md += "|--------|----------|------------|-----------|-----------|-----------|\n"

    for model_name, metrics in results_dict.items():
        acc = f"{metrics.get('accuracy', 0) * 100:.1f}%"
        prec = f"{metrics.get('precision', 0) * 100:.1f}%"

        roc_auc = metrics.get('roc_auc', None)
        sensitivity = f"{metrics.get('sensitivity', 0) * 100:.1f}%"
        specificity = f"{metrics.get('specificity', 0) * 100:.1f}%"
        if isinstance(roc_auc, dict):
            # Average over classes
            auc_val = np.mean(list(roc_auc.values()))
        else:
            auc_val = roc_auc

        auc_str = f"{auc_val * 100:.1f}%" if auc_val is not None else "N/A"

        table_md += f"| {model_name} | {acc} | {auc_str} | {prec} | {sensitivity} | {specificity} |\n"

    return table_md


In [10]:
markdown_output = generate_markdown_table(cleaned_results)
print(markdown_output)

| Model | Accuracy | AUC (avg) | Precision | Sensitivity | Specificity
|--------|----------|------------|-----------|-----------|-----------|
| Frequency | 93.8% | 98.9% | 95.5% | 100.0% | 92.6% |
| Proximity | 94.7% | 86.0% | 95.0% | 66.7% | 100.0% |
| Proximity_1d | 92.0% | 88.3% | 92.7% | 50.0% | 100.0% |
| G-score | 90.3% | 90.8% | 91.7% | 83.3% | 91.6% |
| Frequency + Proximity | 98.2% | 99.4% | 98.3% | 88.9% | 100.0% |
| Frequency + G-score | 98.2% | 99.8% | 98.4% | 100.0% | 97.9% |
| Combined | 97.3% | 99.6% | 97.3% | 88.9% | 98.9% |

